# F1 Intelligence ML - model analysis

Inspect validation model comparisons, test metrics and feature importance.
Run `python scripts/train_all_models.py` and
`python scripts/evaluate_all_models.py` first.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import pandas as pd
import matplotlib.pyplot as plt

import config

report = json.loads((config.REPORTS_DIR / 'evaluation_report.json').read_text(encoding='utf-8'))
for key, value in report['tasks'].items():
    print(key, value.get('algorithm'), value.get('metrics'))

In [ ]:
comparison = json.loads((config.TASK_DIRECTORIES['winner'] / 'after_qualifying' / 'comparison.json').read_text(encoding='utf-8'))
rows = []
for candidate in comparison['candidates']:
    rows.append({
        'algorithm': candidate['algorithm'],
        'score': candidate['score'],
        'log_loss': candidate['metrics'].get('log_loss'),
        'accuracy': candidate['metrics'].get('accuracy'),
    })
pd.DataFrame(rows).sort_values('score', ascending=False)

In [ ]:
importance = pd.read_csv(config.TASK_DIRECTORIES['classification'] / 'after_qualifying' / 'feature_importance.csv')
importance.head(20).set_index('feature')['importance'].plot(kind='barh', figsize=(8, 6))
plt.gca().invert_yaxis()
plt.title('Top classification features')
plt.tight_layout()
plt.show()

In [ ]:
# Inspect the last saved race prediction (if any).
prediction_files = sorted(config.PREDICTIONS_DIR.glob('*/prediction.json'))
if prediction_files:
    payload = json.loads(prediction_files[-1].read_text(encoding='utf-8'))
    print(prediction_files[-1])
    print(json.dumps(payload['race_info'], indent=2))
    print(json.dumps(payload['summary']['winner'], indent=2)[:2000])
else:
    print('No prediction found yet; run scripts/predict_race.py.')